<a href="https://colab.research.google.com/github/di-claro-batera/faculdade_data_science/blob/main/genetico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
import numpy as np
import random

from datetime import datetime

In [58]:
#parametros
n_cities = 6
n_population = 20
mutation_rate = 0.3

In [69]:
# Gera coordenadas aleatórias para as cidades
coordinates_list = [[x, y] for x, y in zip(np.random.randint(0, 100, n_cities), np.random.randint(0, 100, n_cities))]
names_list = np.array(['CidadeA', 'CidadeB', 'CidadeC', 'CidadeD', 'CidadeE', 'CidadeF'])
cities_dict = {x: y for x, y in zip(names_list, coordinates_list)}

# Função para calcular a distância euclidiana entre duas coordenadas
def compute_city_distance_coordinates(a, b):
    return ((a[0] - b[0])**2 + (a[1] - b[1])**2)**0.5

# Função para calcular a distância entre duas cidades usando seus nomes e o dicionário de coordenadas
def compute_city_distance_names(city_a, city_b, cities_dict):
    return compute_city_distance_coordinates(cities_dict[city_a], cities_dict[city_b])

# Função para calcular distâncias fixas (NÃO SERÁ USADA NA AVALIAÇÃO DE FITNESS CORRIGIDA)
def compute_city_distance_coordinates_fixo(a, b):
    if a == 'CidadeA' and b == 'CidadeB':
        return 1
    if a == 'CidadeA' and b == 'CidadeC':
        return 2
    if a == 'CidadeA' and b == 'CidadeD':
        return 3
    if a == 'CidadeA' and b == 'CidadeE':
        return 1
    if a == 'CidadeA' and b == 'CidadeF':
        return 2
    if a == 'CidadeB' and b == 'CidadeA':
        return 1
    return 1

In [70]:
# Função para calcular distâncias fixas usando nomes (TAMBÉM NÃO SERÁ USADA)
def compute_city_distance_names_fixo(city_a, city_b, cities_dict):
    return compute_city_distance_coordinates_fixo(city_a, city_b)

cities_dict

{np.str_('CidadeA'): [np.int64(69), np.int64(33)],
 np.str_('CidadeB'): [np.int64(20), np.int64(5)],
 np.str_('CidadeC'): [np.int64(82), np.int64(10)],
 np.str_('CidadeD'): [np.int64(94), np.int64(22)],
 np.str_('CidadeE'): [np.int64(81), np.int64(0)],
 np.str_('CidadeF'): [np.int64(19), np.int64(64)]}

In [71]:
# Função para gerar a população inicial de soluções aleatórias
def genesis(city_list, n_population):
    population_set = []
    for i in range(n_population):
        # Gera aleatoriamente uma nova ordem das cidades
        sol_i = city_list[np.random.choice(list(range(n_cities)), n_cities, replace=False)]
        population_set.append(sol_i)
    return np.array(population_set)

# Cria a população inicial
population_set = genesis(names_list, n_population)
population_set

array([['CidadeA', 'CidadeC', 'CidadeF', 'CidadeE', 'CidadeB', 'CidadeD'],
       ['CidadeB', 'CidadeA', 'CidadeD', 'CidadeC', 'CidadeF', 'CidadeE'],
       ['CidadeD', 'CidadeB', 'CidadeC', 'CidadeE', 'CidadeA', 'CidadeF'],
       ['CidadeE', 'CidadeA', 'CidadeD', 'CidadeF', 'CidadeC', 'CidadeB'],
       ['CidadeC', 'CidadeB', 'CidadeE', 'CidadeA', 'CidadeD', 'CidadeF'],
       ['CidadeE', 'CidadeF', 'CidadeB', 'CidadeA', 'CidadeD', 'CidadeC'],
       ['CidadeA', 'CidadeF', 'CidadeD', 'CidadeB', 'CidadeC', 'CidadeE'],
       ['CidadeA', 'CidadeC', 'CidadeE', 'CidadeF', 'CidadeB', 'CidadeD'],
       ['CidadeB', 'CidadeC', 'CidadeE', 'CidadeA', 'CidadeF', 'CidadeD'],
       ['CidadeF', 'CidadeD', 'CidadeB', 'CidadeC', 'CidadeE', 'CidadeA'],
       ['CidadeA', 'CidadeF', 'CidadeE', 'CidadeB', 'CidadeC', 'CidadeD'],
       ['CidadeA', 'CidadeD', 'CidadeE', 'CidadeF', 'CidadeC', 'CidadeB'],
       ['CidadeE', 'CidadeB', 'CidadeA', 'CidadeD', 'CidadeF', 'CidadeC'],
       ['CidadeF', 'Cidad

In [72]:
# Função para avaliar o fitness de uma rota (quanto menor, melhor)
def fitness_eval(city_list, cities_dict):
    total_distance = 0
    for i in range(n_cities - 1):
        city_a = city_list[i]
        city_b = city_list[i + 1]
        total_distance += compute_city_distance_names_fixo(city_a, city_b, cities_dict)
    return total_distance

In [73]:
# Função para obter o fitness de toda a população
def get_all_fitnes(population_set, cities_dict):
    fitnes_list = np.zeros(n_population)
    for i in range(n_population):
        fitnes_list[i] = fitness_eval(population_set[i], cities_dict)
    return fitnes_list

# Avalia o fitness da população inicial
fitnes_list = get_all_fitnes(population_set, cities_dict)
fitnes_list

array([6., 7., 6., 7., 7., 7., 6., 6., 6., 5., 6., 7., 7., 5., 7., 5., 7.,
       5., 5., 6.])

In [74]:
# Função para selecionar os progenitores para a reprodução (baseado na probabilidade inversa do fitness - quanto menor o fitness, maior a probabilidade)
def progenitor_selection(population_set, fitnes_list):
    # Inverte o fitness para que menores distâncias tenham maiores probabilidades
    inverse_fitness = 1 / (fitnes_list + 1e-10)  # Adiciona um pequeno valor para evitar divisão por zero
    total_inverse_fit = inverse_fitness.sum()
    prob_list = inverse_fitness / total_inverse_fit

    progenitor_list_a_idx = np.random.choice(list(range(len(population_set))), len(population_set), p=prob_list, replace=True)
    progenitor_list_b_idx = np.random.choice(list(range(len(population_set))), len(population_set), p=prob_list, replace=True)

    progenitor_list_a = [population_set[idx] for idx in progenitor_list_a_idx]
    progenitor_list_b = [population_set[idx] for idx in progenitor_list_b_idx]

    return np.array([progenitor_list_a, progenitor_list_b])

# Seleciona os progenitores
progenitor_list = progenitor_selection(population_set, fitnes_list)
progenitor_list[0][2]

array(['CidadeA', 'CidadeF', 'CidadeE', 'CidadeB', 'CidadeC', 'CidadeD'],
      dtype='<U7')

In [75]:
# Função para realizar o crossover (recombinação) entre dois progenitores (usando uma forma simples de crossover de ordem)
def mate_progenitors(prog_a, prog_b):
    start = np.random.randint(0, n_cities)
    end = np.random.randint(start + 1, n_cities + 1)
    offspring = np.array([None] * n_cities)
    offspring[start:end] = prog_a[start:end]
    current_index = 0
    for city in prog_b:
        if city not in offspring:
            while offspring[current_index] is not None:
                current_index += 1
            offspring[current_index] = city
    return offspring

# Função para realizar o crossover em toda a população de progenitores
def mate_population(progenitor_list):
    new_population_set = []
    for i in range(progenitor_list.shape[1]):
        prog_a, prog_b = progenitor_list[0][i], progenitor_list[1][i]
        offspring = mate_progenitors(prog_a, prog_b)
        new_population_set.append(offspring)
    return np.array(new_population_set)

# Realiza o crossover
new_population_set = mate_population(progenitor_list)
new_population_set[0]

array([np.str_('CidadeD'), np.str_('CidadeB'), 'CidadeC', 'CidadeE',
       'CidadeA', 'CidadeF'], dtype=object)

In [76]:
# Função para aplicar mutação a um filho (troca aleatoriamente duas cidades)
def mutate_offspring(offspring):
    for _ in range(int(n_cities * mutation_rate)):
        a = np.random.randint(0, n_cities)
        b = np.random.randint(0, n_cities)
        offspring[a], offspring[b] = offspring[b], offspring[a]
    return offspring

# Função para aplicar mutação a toda a nova população
def mutate_population(new_population_set):
    mutated_pop = []
    for offspring in new_population_set:
        mutated_pop.append(mutate_offspring(offspring))
    return np.array(mutated_pop)

# Aplica a mutação
mutated_pop = mutate_population(new_population_set)
mutated_pop[0]

array(['CidadeA', np.str_('CidadeB'), 'CidadeC', 'CidadeE',
       np.str_('CidadeD'), 'CidadeF'], dtype=object)

In [77]:
# Loop principal do algoritmo genético
best_solution = [-1, np.inf, np.array([])]
for i in range(10000):
    if i % 100 == 0:
        print(f"Geração: {i}, Melhor Fitness: {best_solution[1]:.2f}, Fitness Médio: {fitnes_list.mean():.2f}, Hora: {datetime.now().strftime('%d/%m/%y %H:%M')}")

    fitnes_list = get_all_fitnes(mutated_pop, cities_dict)

    if fitnes_list.min() < best_solution[1]:
        best_solution[0] = i
        best_solution[1] = fitnes_list.min()
        best_solution[2] = np.array(mutated_pop)[fitnes_list.argmin() == np.arange(len(mutated_pop))]

    progenitor_list = progenitor_selection(mutated_pop, fitnes_list)
    new_population_set = mate_population(progenitor_list)
    mutated_pop = mutate_population(new_population_set)

Geração: 0, Melhor Fitness: inf, Fitness Médio: 6.15, Hora: 07/05/25 12:19
Geração: 100, Melhor Fitness: 5.00, Fitness Médio: 5.80, Hora: 07/05/25 12:19
Geração: 200, Melhor Fitness: 5.00, Fitness Médio: 5.55, Hora: 07/05/25 12:19
Geração: 300, Melhor Fitness: 5.00, Fitness Médio: 5.80, Hora: 07/05/25 12:19
Geração: 400, Melhor Fitness: 5.00, Fitness Médio: 5.25, Hora: 07/05/25 12:19
Geração: 500, Melhor Fitness: 5.00, Fitness Médio: 5.55, Hora: 07/05/25 12:19
Geração: 600, Melhor Fitness: 5.00, Fitness Médio: 5.85, Hora: 07/05/25 12:19
Geração: 700, Melhor Fitness: 5.00, Fitness Médio: 5.90, Hora: 07/05/25 12:19
Geração: 800, Melhor Fitness: 5.00, Fitness Médio: 6.05, Hora: 07/05/25 12:19
Geração: 900, Melhor Fitness: 5.00, Fitness Médio: 5.75, Hora: 07/05/25 12:19
Geração: 1000, Melhor Fitness: 5.00, Fitness Médio: 5.55, Hora: 07/05/25 12:19
Geração: 1100, Melhor Fitness: 5.00, Fitness Médio: 5.70, Hora: 07/05/25 12:19
Geração: 1200, Melhor Fitness: 5.00, Fitness Médio: 5.45, Hora: 0

In [78]:
print("\nMelhor Solução Encontrada:")
print(f"Geração: {best_solution[0]}")
print(f"Distância Total: {best_solution[1]:.2f}")
print(f"Rota: {[str(city) for city in best_solution[2][0]] if best_solution[2].size > 0 else 'Nenhuma solução encontrada'}")
print("\nCoordenadas das Cidades:")
for city, coords in cities_dict.items():
    print(f"{str(city)}: {[int(coord) for coord in coords]}")


Melhor Solução Encontrada:
Geração: 0
Distância Total: 5.00
Rota: ['CidadeA', 'CidadeB', 'CidadeC', 'CidadeE', 'CidadeD', 'CidadeF']

Coordenadas das Cidades:
CidadeA: [69, 33]
CidadeB: [20, 5]
CidadeC: [82, 10]
CidadeD: [94, 22]
CidadeE: [81, 0]
CidadeF: [19, 64]
